# Boosting Regression - AdaboostRegressor 

In [214]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_selection import chi2


## Carregar os Dados

In [215]:
df_costs = pd.read_csv('./dataset/employees.csv')
df_costs.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   age              1338 non-null   int64  
 1   sex              1338 non-null   str    
 2   bmi              1338 non-null   float64
 3   children         1338 non-null   int64  
 4   smoker           1338 non-null   int64  
 5   region           1338 non-null   str    
 6   medical charges  1338 non-null   float64
dtypes: float64(2), int64(3), str(2)
memory usage: 73.3 KB


In [216]:
df_costs.head(10)

,age,sex,bmi,children,smoker,region,medical charges
0,19,female,27.900,0,1,southwest,16884.92400
1,18,male,33.770,1,0,southeast,1725.55230
2,28,male,33.000,3,0,southeast,4449.46200
3,33,male,22.705,0,0,northwest,21984.47061
4,32,male,28.880,0,0,northwest,3866.85520
5,31,female,25.740,0,0,southeast,3756.62160
6,46,female,33.440,1,0,southeast,8240.58960
7,37,female,27.740,3,0,northwest,7281.50560
8,37,male,29.830,2,0,northeast,6406.41070
9,60,female,25.840,0,0,northwest,28923.13692


## Ler preprocessor

In [217]:
preprocessor: ColumnTransformer = joblib.load('preprocessor.pkl')

## Preparação de Dados

In [218]:
X = df_costs.drop(columns=['medical charges'])
y = df_costs['medical charges']

numeric_features = X.select_dtypes('number').columns.tolist()
cat_features = X.select_dtypes('str').columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=51)

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [219]:
X_train.shape

(1070, 10)

In [220]:
X_test.shape

(268, 10)

## Treinamento do Modelo

In [221]:
boosting_model = AdaBoostRegressor(
  estimator=LinearRegression(),
  n_estimators=50,
  random_state=51,
  learning_rate=0.01,
)

In [222]:
boosting_model.fit(X_train, y_train)

,"estimator estimator: object, default=NoneThe base estimator from which the boosted ensemble is built.If ``None``, then the base estimator is:class:`~sklearn.tree.DecisionTreeRegressor` initialized with`max_depth=3`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",LinearRegression()
,"learning_rate learning_rate: float, default=1.0Weight applied to each regressor at each boosting iteration. A higherlearning rate increases the contribution of each regressor. There isa trade-off between the `learning_rate` and `n_estimators` parameters.Values must be in the range `(0.0, inf)`.",0.01
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given at each `estimator` at eachboosting iteration.Thus, it is only used when `estimator` exposes a `random_state`.In addition, it controls the bootstrap of the weights used to train the`estimator` at each boosting iteration.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",51
,"n_estimators n_estimators: int, default=50The maximum number of estimators at which boosting is terminated.In case of perfect fit, the learning procedure is stopped early.Values must be in the range `[1, inf)`.",50
,"loss loss: {'linear', 'square', 'exponential'}, default='linear'The loss function to use when updating the weights after eachboosting iteration.",'linear'
Name,Type,Value
estimator_ estimator_: estimatorThe base estimator from which the ensemble is grown... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,LinearRegression,LinearRegression()
estimator_errors_ estimator_errors_: ndarray of floatsRegression error for each estimator in the boosted ensemble.,"ndarray[float64](50,)","[0.13,0.14,0.13,...,0.16,0.15,0.16]"
estimator_weights_ estimator_weights_: ndarray of floatsWeights for each estimator in the boosted ensemble.,"ndarray[float64](50,)","[0.02,0.02,0.02,...,0.02,0.02,0.02]"
estimators_ estimators_: list of regressorsThe collection of fitted sub-estimators.,list,"[LinearRegression(), LinearRegression(), LinearRegression(), LinearRegression(), ...]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,10


## Análise de Métricas
- Erro absoluto médio de 5400 dolares é um erro muito considerável.

In [223]:
y_pred = boosting_model.predict(X_test)

In [224]:
mae = mean_absolute_error(y_pred=y_pred, y_true=y_test)
rmse = root_mean_squared_error(y_pred=y_pred, y_true=y_test)
r2 = r2_score(y_pred=y_pred, y_true=y_test)

In [225]:
print(f"Mean Absolute Error: {mae}")
print(f"Root Mean Squared Error: {rmse}")
print(f"R2-Score: {r2}")

Mean Absolute Error: 4596.405705344629
Root Mean Squared Error: 6606.293157423567
R2-Score: 0.7490081386095852


## Importância das Features

In [226]:
coefs = [estimator.coef_ for estimator in boosting_model.estimators_]

In [227]:
importances = np.mean(np.abs(coefs), axis=0)

In [228]:
importances_percentage = importances / np.sum(importances)

In [229]:
# Obter nome das features
feature_names = preprocessor.get_feature_names_out()
feature_names

array(['num__age', 'num__bmi', 'num__children', 'num__smoker',
       'cat__sex_female', 'cat__sex_male', 'cat__region_northeast',
       'cat__region_northwest', 'cat__region_southeast',
       'cat__region_southwest'], dtype=object)

In [230]:
# Dataset contendo importância percentual e nome das features
df_features = pd.DataFrame({ 'importance': importances_percentage, 'feature': feature_names})

In [231]:
df_features.sort_values(by='importance', ascending=True, inplace=True)

In [232]:
px.bar(
  df_features,
  x='importance',
  y='feature',
  orientation='h',
  title="Importância Percentual das Features",
)

## Propriedades do Modelo

### Erros dos Estimadores
- Note, o modelo para ao convergir, logo se o modelo atingir a não convergência antes de o número total de estimadores, esses estimadores com erro = 1 são inúteis, podendo ser descartados.

In [233]:
boosting_model.estimator_errors_

array([0.1325525 , 0.13628964, 0.13089805, 0.13518575, 0.13637757,
       0.13796994, 0.13713767, 0.13598896, 0.13926219, 0.13609188,
       0.1414793 , 0.13928107, 0.14268834, 0.14165246, 0.14175235,
       0.1391914 , 0.14428085, 0.14617396, 0.14060611, 0.13842243,
       0.14060113, 0.14505285, 0.14624795, 0.14327705, 0.15344559,
       0.14797373, 0.14437861, 0.14821357, 0.14489608, 0.14491297,
       0.15460493, 0.15312197, 0.15099891, 0.14575804, 0.15034644,
       0.14882261, 0.15636585, 0.15266722, 0.15821031, 0.14838691,
       0.15647719, 0.15662499, 0.15335604, 0.14760064, 0.15552772,
       0.15317045, 0.16051104, 0.15694728, 0.1541718 , 0.15775678])

### Peso dos Estimadores
- Note: modelos que tem erro de 1 recebem peso = 0 para não serem usados.

In [234]:
boosting_model.estimator_weights_

array([0.01878576, 0.01846455, 0.01893042, 0.01855865, 0.01845708,
       0.01832254, 0.0183927 , 0.01849012, 0.01821432, 0.01848136,
       0.01803057, 0.01821274, 0.01793139, 0.01801632, 0.01800811,
       0.01822022, 0.0178018 , 0.0176493 , 0.01810265, 0.01828455,
       0.01810306, 0.01773941, 0.01764337, 0.01788334, 0.01707828,
       0.01750583, 0.01779389, 0.01748682, 0.01775206, 0.0177507 ,
       0.01698931, 0.01710322, 0.01726788, 0.01768267, 0.01731886,
       0.01743865, 0.0168552 , 0.01713833, 0.01671605, 0.01747309,
       0.01684677, 0.01683557, 0.01708518, 0.01753545, 0.01691888,
       0.01709948, 0.01654431, 0.0168112 , 0.01702249, 0.01675014])